In [ ]:
!pip install tensorflow opencv-python pandas pillow scikit-learn


In [ ]:
import pandas as pd
import numpy as np
import os
import cv2
from PIL import Image

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical


In [ ]:
# =========================
# 4. LOAD DATASET
# =========================
import os
import pandas as pd

columns = [
    "Timestamp","CAN_ID","DLC",
    "D0","D1","D2","D3","D4","D5","D6","D7",
    "Class"
]

file_path = "DoS_dataset.csv"

if os.path.exists(file_path):
    df = pd.read_csv(file_path, names=columns)
    # LIMIT DATA to avoid memory issues
    df = df.head(1000000)
    print("Dataset loaded. Size:", df.shape)
    display(df.head())
else:
    print("❌ DoS_dataset.csv not found. Please run the upload cell above first.")

Dataset loaded. Size: (1000000, 12)


,Timestamp,CAN_ID,DLC,D0,D1,D2,D3,D4,D5,D6,D7,Class
0,1.478198e+09,0316,8,05,21,68,09,21,21,00,6f,R
1,1.478198e+09,018f,8,fe,5b,00,00,00,3c,00,00,R
2,1.478198e+09,0260,8,19,21,22,30,08,8e,6d,3a,R
3,1.478198e+09,02a0,8,64,00,9a,1d,97,02,bd,00,R
4,1.478198e+09,0329,8,40,bb,7f,14,11,20,00,14,R


In [ ]:
# =========================
# CLEAN + PREPROCESS
# =========================

# Ensure df exists before proceeding
if 'df' in locals():
    # strip spaces
    df = df.astype(str).apply(lambda x: x.str.strip())

    # FIX LABELS
    df['Class'] = df['Class'].replace({'R':0, 'T':1})
    df['Class'] = pd.to_numeric(df['Class'], errors='coerce')
    df = df.dropna(subset=['Class'])
    df['Class'] = df['Class'].astype(int)

    # SAFE HEX CONVERSION
    def safe_hex(x):
        try:
            return int(str(x), 16)
        except:
            return 0

    df['CAN_ID'] = df['CAN_ID'].apply(safe_hex)
    for col in ['D0','D1','D2','D3','D4','D5','D6','D7']:
        df[col] = df[col].apply(safe_hex)

    df['DLC'] = pd.to_numeric(df['DLC'], errors='coerce')
    df = df.fillna(0)

    feature_cols = ['CAN_ID','DLC','D0','D1','D2','D3','D4','D5','D6','D7']
    for col in feature_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # Prepare features and labels for the image conversion step
    features = df[feature_cols].values
    labels = to_categorical(df['Class'].values, num_classes=2)
    print("Preprocessing complete. Features and labels ready.")
else:
    print("Please load the dataset successfully first.")

Preprocessing complete. Features and labels ready.


In [ ]:
df = df.dropna(subset=['Class'])

In [ ]:
image_dir = "/content/can_images"
os.makedirs(image_dir, exist_ok=True)

for i in range(len(features)-25):  

    patch = features[i:i+81].flatten()

    if len(patch) < 243:
        continue

    img = np.array(patch[:243], dtype=np.uint8).reshape(9,9,3)

    img = cv2.resize(img,(224,224))

    label = np.argmax(labels[i])

    class_dir = os.path.join(image_dir,str(label))
    os.makedirs(class_dir, exist_ok=True)

    Image.fromarray(img).save(f"{class_dir}/{i}.png")

In [ ]:
datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_data = datagen.flow_from_directory(
    image_dir,
    target_size=(224,224),
    batch_size=128,
    class_mode='categorical',
    subset='training'
)

val_data = datagen.flow_from_directory(
    image_dir,
    target_size=(224,224),
    batch_size=128,
    class_mode='categorical',
    subset='validation'
)

Found 793899 images belonging to 2 classes.
Found 198474 images belonging to 2 classes.


In [ ]:
base_model = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

for layer in base_model.layers:
    layer.trainable = False


58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
x = base_model.output
x = GlobalAveragePooling2D()(x)

x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)

output = Dense(2, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=output)


In [ ]:
model.compile(
    optimizer=Adam(),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)


In [ ]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=100
)

In [ ]:
# =========================
# 13. EVALUATE
# =========================
loss, acc = model.evaluate(val_data)
print("Final Accuracy:", acc)

# =========================
# 14. SAVE MODEL
# =========================
model.save("vgg16_dos_fast_model.h5")